### Load in the Model (Llama-3.1-8B-Instruct)

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", dtype="float16", device_map="cuda")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


### Generating Activations

In [23]:
import pandas as pd

# Load in the Factual Datasets
F0_train, F0_test = pd.read_csv("../dataset/F0_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F0_test.csv")[["statement", "label"]]
F1_train, F1_test = pd.read_csv("../dataset/F1_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F1_test.csv")[["statement", "label"]]
F2_train, F2_test = pd.read_csv("../dataset/F2_train.csv"), pd.read_csv("../dataset/F2_test.csv")
F3_train, F3_test = pd.read_csv("../dataset/F3_train.csv"), pd.read_csv("../dataset/F3_test.csv")
F4_train, F4_test = pd.read_csv("../dataset/F4_train.csv"), pd.read_csv("../dataset/F4_test.csv")
F5_train, F5_test = pd.read_csv("../dataset/F5_train.csv"), pd.read_csv("../dataset/F5_test.csv")

# Load in the Arithmatic Statements
A1_train, A1_test = pd.read_csv("../dataset/A1_train.csv"), pd.read_csv("../dataset/A1_test.csv")
A2_train, A2_test = pd.read_csv("../dataset/A2_train.csv"), pd.read_csv("../dataset/A2_test.csv")
A3_train, A3_test = pd.read_csv("../dataset/A3_train.csv"), pd.read_csv("../dataset/A3_test.csv")


In [24]:
datasets = {
    "F0_train": F0_train, "F0_test": F0_test,
    "F1_train": F1_train, "F1_test": F1_test,
    "F2_train": F2_train, "F2_test": F2_test,
    "F3_train": F3_train, "F3_test": F3_test,
    "F4_train": F4_train, "F4_test": F4_test,
    "F5_train": F5_train, "F5_test": F5_test,
    "A1_train": A1_train, "A1_test": A1_test,
    "A2_train": A2_train, "A2_test": A2_test,
    "A3_train": A3_train, "A3_test": A3_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)}")


F0_train: 1194
F0_test: 512
F1_train: 1194
F1_test: 512
F2_train: 56
F2_test: 24
F3_train: 1400
F3_test: 600
F4_train: 1400
F4_test: 600
F5_train: 1400
F5_test: 600
A1_train: 700
A1_test: 300
A2_train: 700
A2_test: 300
A3_train: 700
A3_test: 300


In [25]:
import torch
from tqdm import tqdm

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def generate_activations(model, statements, layer, with_chat_template=True, batch_size=16):
    statements = list(statements)
    final_token_activations = []

    for i in range(0, len(statements), batch_size):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in statements_temp]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            inputs = tokenizer(statements_temp, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        final_token_activation = outputs.hidden_states[layer][:, -1, :]
        final_token_activations.append(final_token_activation.cpu())

    return torch.cat(final_token_activations, dim=0)

In [26]:
for dataset in tqdm([F0_test, F0_train,
                     F1_train, F1_test,
                     F2_train, F2_test,
                     F3_train, F3_test,
                     F4_test, F4_train,
                     F5_train, F5_test, 
                     A1_test, A1_train,
                     A2_test, A2_train, 
                     A3_test, A3_train]):
    activations = generate_activations(model, dataset["statement"], 16, with_chat_template=True, batch_size=16)
    dataset["activations_chat"] = list(activations)
    activations = generate_activations(model, dataset["statement"], 16, with_chat_template=False, batch_size=16)
    dataset["activations"] = list(activations)

100%|██████████| 18/18 [02:09<00:00,  7.19s/it]


In [27]:
F0_test["activations"]

0      [tensor(-0.0815, dtype=torch.float16), tensor(...
1      [tensor(0.0090, dtype=torch.float16), tensor(-...
2      [tensor(0.0620, dtype=torch.float16), tensor(0...
3      [tensor(-0.0927, dtype=torch.float16), tensor(...
4      [tensor(-0.1384, dtype=torch.float16), tensor(...
                             ...                        
507    [tensor(-0.0697, dtype=torch.float16), tensor(...
508    [tensor(-0.0650, dtype=torch.float16), tensor(...
509    [tensor(0.0288, dtype=torch.float16), tensor(0...
510    [tensor(-0.0469, dtype=torch.float16), tensor(...
511    [tensor(0.0591, dtype=torch.float16), tensor(0...
Name: activations, Length: 512, dtype: object

### Training the Model and Extracting AUROC Scores

In [28]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def train_probe_pytorch(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels

    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()


    # Mean-center using only the training mean
    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]

    # THIS is the entire "model": one linear layer, no bias.
    # w(x) = w^T x, no offset term -> passes through the origin.
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)

    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()  # sigmoid + binary cross-entropy, combined for numerical stability

    for step in range(1000):
        optimizer.zero_grad()
        logits = probe(X_train_t).squeeze(-1)   # w^T x for every example
        loss = loss_fn(logits, y_train_t)
        loss.backward()
        optimizer.step()

    # Evaluate
    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()
    auroc = roc_auc_score(y_test, test_logits)

    return probe, train_mean, auroc

In [29]:
train_probe_pytorch((F0_train["activations"], F0_test["activations"]), (F0_train["label"], F0_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.04522375,  0.02557962,  0.06479687, ...,  0.08218045,
         0.15706071,  0.1257306 ], shape=(4096,), dtype=float32),
 0.9993133544921875)

In [30]:
train_probe_pytorch((F1_train["activations"], F1_test["activations"]), (F1_train["label"], F1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.09417498,  0.044316  ,  0.15139177, ...,  0.03234628,
         0.05480164, -0.07801153], shape=(4096,), dtype=float32),
 1.0)

In [31]:
train_probe_pytorch((F2_train["activations"], F2_test["activations"]), (F2_train["label"], F2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.05057464, -0.04331861, -0.08072921, ...,  0.04919324,
         0.01248905, -0.03941618], shape=(4096,), dtype=float32),
 0.9513888888888888)

In [32]:
train_probe_pytorch((F3_train["activations"], F3_test["activations"]), (F3_train["label"], F3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.03529509, -0.01319063,  0.04801562, ..., -0.00054901,
        -0.00465954,  0.12306307], shape=(4096,), dtype=float32),
 0.9999999999999999)

In [33]:
train_probe_pytorch((F4_train["activations"], F4_test["activations"]), (F4_train["label"], F4_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.05588087,  0.10826533, -0.00557619, ...,  0.0032822 ,
        -0.16908574,  0.12390584], shape=(4096,), dtype=float32),
 0.8068777777777778)

In [34]:
train_probe_pytorch((F5_train["activations"], F5_test["activations"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.06159449,  0.04821751, -0.13252231, ...,  0.02939573,
        -0.13362476,  0.13972335], shape=(4096,), dtype=float32),
 0.9604777777777778)

In [35]:
train_probe_pytorch((A1_train["activations"], A1_test["activations"]), (A1_train["label"], A1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.10054926, -0.1698112 ,  0.06591169, ...,  0.0347801 ,
         0.05990821,  0.00083482], shape=(4096,), dtype=float32),
 0.6446222222222222)

In [36]:
train_probe_pytorch((A2_train["activations"], A2_test["activations"]), (A2_train["label"], A2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.11967925, -0.09901559,  0.05275749, ..., -0.03800526,
         0.00885455, -0.03281679], shape=(4096,), dtype=float32),
 0.5796)

In [37]:
train_probe_pytorch((A3_train["activations"], A3_test["activations"]), (A3_train["label"], A3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.15804632, -0.10450189,  0.04143262, ..., -0.03563659,
         0.02630921, -0.02770448], shape=(4096,), dtype=float32),
 0.5740000000000001)

In [38]:
train_probe_pytorch((F0_train["activations_chat"], F0_test["activations_chat"]), (F0_train["label"], F0_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.00403655,  0.01345949, -0.13136557, ..., -0.02833014,
        -0.0637959 ,  0.01483763], shape=(4096,), dtype=float32),
 0.999755859375)

In [39]:
train_probe_pytorch((F1_train["activations_chat"], F1_test["activations_chat"]), (F1_train["label"], F1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.06741486,  0.0085224 , -0.10556439, ..., -0.0626237 ,
        -0.08024353,  0.03838613], shape=(4096,), dtype=float32),
 1.0)

In [40]:
train_probe_pytorch((F2_train["activations_chat"], F2_test["activations_chat"]), (F2_train["label"], F2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.02336829,  0.09497847, -0.02521488, ..., -0.12140628,
        -0.03836495,  0.02890069], shape=(4096,), dtype=float32),
 0.9652777777777778)

In [41]:
train_probe_pytorch((F3_train["activations_chat"], F3_test["activations_chat"]), (F3_train["label"], F3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.14855923,  0.08854906,  0.01523246, ..., -0.15391591,
        -0.21092477,  0.08380454], shape=(4096,), dtype=float32),
 1.0)

In [42]:
train_probe_pytorch((F4_train["activations_chat"], F4_test["activations_chat"]), (F4_train["label"], F4_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.17785734,  0.17865627,  0.00092682, ..., -0.00814655,
        -0.08403013,  0.03057657], shape=(4096,), dtype=float32),
 0.8224444444444445)

In [43]:
train_probe_pytorch((F5_train["activations_chat"], F5_test["activations_chat"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.19156538,  0.15625471, -0.03895089, ..., -0.02957476,
        -0.06541162,  0.02094537], shape=(4096,), dtype=float32),
 0.9698555555555556)

In [44]:
train_probe_pytorch((A1_train["activations_chat"], A1_test["activations_chat"]), (A1_train["label"], A1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.11695445,  0.06046747,  0.05797121, ..., -0.06279703,
        -0.04388838,  0.04010627], shape=(4096,), dtype=float32),
 0.6445777777777778)

In [45]:
train_probe_pytorch((A2_train["activations_chat"], A2_test["activations_chat"]), (A2_train["label"], A2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.18006985,  0.08285514,  0.07934352, ..., -0.03520111,
        -0.03425208, -0.04565858], shape=(4096,), dtype=float32),
 0.5273333333333333)

In [46]:
train_probe_pytorch((A3_train["activations_chat"], A3_test["activations_chat"]), (A3_train["label"], A3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.20059204,  0.08768764,  0.06997777, ..., -0.03213523,
        -0.02398991, -0.06627693], shape=(4096,), dtype=float32),
 0.5030222222222221)

We find that chat-template application has negligible effect on simple factual tasks (F0, F1) and simple arithmetic (A1), a small positive effect on complex factual tasks (F4), but substantially degrades performance on the hardest arithmetic task (A3, dropping to near-chance). Given that raw-text prompting produces equal-or-better performance across all tested tasks except F4, and that Poulis's prompt-template examples (Appendix A.3) are shown without chat-formatting, we adopt no-chat-template as our primary methodology, consistent with the likely convention used in the source paper.